In [94]:
import pandas as pd

mises_refs = pd.read_csv("../data/raw/scopus_papers_citing_mises.csv")
hayek_refs = pd.read_csv("../data/raw/scopus_papers_citing_hayek.csv")

In [95]:
mises_refs.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7458 entries, 0 to 7457
Data columns (total 23 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   Authors            7439 non-null   object
 1   Author full names  7439 non-null   object
 2   Author(s) ID       7439 non-null   object
 3   Title              7458 non-null   object
 4   Year               7458 non-null   int64 
 5   Source title       7458 non-null   object
 6   Volume             4832 non-null   object
 7   Issue              4164 non-null   object
 8   Art. No.           404 non-null    object
 9   Page start         6913 non-null   object
 10  Page end           6908 non-null   object
 11  Cited by           7458 non-null   int64 
 12  DOI                6262 non-null   object
 13  Link               7458 non-null   object
 14  Abstract           7458 non-null   object
 15  Author Keywords    3669 non-null   object
 16  Index Keywords     1365 non-null   object


In [96]:
# Adiciona flags antes de unificar
mises_refs['cite_mises'] = True
hayek_refs['cite_hayek'] = True

# Merge outer por Title + Source title, trazendo References do hayek também
all_refs = pd.merge(
    mises_refs,
    hayek_refs[['Title', 'Source title', 'cite_hayek', 'References']],
    on=['Title', 'Source title'],
    how='outer',
    suffixes=('', '_hayek')
)

# Garante References preenchido: prioriza o do mises, cai pro hayek se vazio
all_refs['References'] = all_refs['References'].combine_first(all_refs['References_hayek'])
all_refs = all_refs.drop(columns=['References_hayek'])

# Preenche os ausentes com False
all_refs['cite_mises'] = all_refs['cite_mises'].fillna(False)
all_refs['cite_hayek'] = all_refs['cite_hayek'].fillna(False)

total      = len(all_refs)
only_mises = (all_refs['cite_mises'] & ~all_refs['cite_hayek']).sum()
only_hayek = (all_refs['cite_hayek'] & ~all_refs['cite_mises']).sum()
ambos      = (all_refs['cite_mises'] &  all_refs['cite_hayek']).sum()
n_mises    = all_refs['cite_mises'].sum()
n_hayek    = all_refs['cite_hayek'].sum()

p_hayek_dado_mises = ambos / n_mises
p_mises_dado_hayek = ambos / n_hayek

print(f"mises_refs: {len(mises_refs):,} linhas")
print(f"hayek_refs: {len(hayek_refs):,} linhas")
print(f"all_refs:   {total:,} linhas")

print(f"\n{'Grupo':<20} {'N':>7} {'%':>7}")
print("-" * 36)
print(f"{'cite_mises only':<20} {only_mises:>7,} {only_mises/total*100:>6.1f}%")
print(f"{'cite_hayek only':<20} {only_hayek:>7,} {only_hayek/total*100:>6.1f}%")
print(f"{'ambos':<20} {ambos:>7,} {ambos/total*100:>6.1f}%")
print("-" * 36)
print(f"{'Total':<20} {total:>7,} {'100.0%':>7}")

print(f"\n{'Probabilidades condicionais'}")
print("-" * 36)
print(f"P(Hayek | Mises): {ambos:,} / {n_mises:,} = {p_hayek_dado_mises*100:.1f}%")
print(f"P(Mises | Hayek): {ambos:,} / {n_hayek:,} = {p_mises_dado_hayek*100:.1f}%")

# Checagem
vazios = all_refs['References'].isna().sum()
print(f"\nReferences vazias após merge: {vazios:,} ({vazios/total*100:.1f}%)")

mises_refs: 7,458 linhas
hayek_refs: 19,998 linhas
all_refs:   24,779 linhas

Grupo                      N       %
------------------------------------
cite_mises only        4,735   19.1%
cite_hayek only       17,272   69.7%
ambos                  2,772   11.2%
------------------------------------
Total                 24,779  100.0%

Probabilidades condicionais
------------------------------------
P(Hayek | Mises): 2,772 / 7,507 = 36.9%
P(Mises | Hayek): 2,772 / 20,044 = 13.8%

References vazias após merge: 0 (0.0%)


C:\Users\pedro\AppData\Local\Temp\ipykernel_19000\1166384598.py:19: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  all_refs['cite_mises'] = all_refs['cite_mises'].fillna(False)
C:\Users\pedro\AppData\Local\Temp\ipykernel_19000\1166384598.py:20: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  all_refs['cite_hayek'] = all_refs['cite_hayek'].fillna(False)


In [97]:
import re

# Média de keywords ANTES da expansão
def count_keywords(kw_string):
    if pd.isna(kw_string) or kw_string == '':
        return 0
    return len([kw for kw in kw_string.split(';') if kw.strip()])

media_antes = all_refs['Author Keywords'].apply(count_keywords).mean()
print(f"Média de keywords por artigo ANTES: {media_antes:.2f}")

# Coleta todas as keywords únicas já existentes no dataset
all_keywords = set()
for kw_string in all_refs['Author Keywords'].dropna():
    for kw in kw_string.split(';'):
        kw = kw.strip().lower()
        if kw and len(kw) > 3:
            all_keywords.add(kw)

print(f"Total de keywords únicas no dataset (>3 chars): {len(all_keywords):,}")

# Compila um único regex com todas as keywords de uma vez
pattern = re.compile(
    r'\b(' + '|'.join(re.escape(kw) for kw in all_keywords) + r')\b'
)

added_total = 0

def expand_keywords(row):
    global added_total

    # Combina Title + Abstract para busca
    title    = row['Title']    if pd.notna(row['Title'])    else ''
    abstract = row['Abstract'] if pd.notna(row['Abstract']) else ''
    text = f"{title} {abstract}".strip().lower()

    if not text:
        return row['Author Keywords']

    # Keywords já existentes no artigo
    existing = set()
    if pd.notna(row['Author Keywords']) and row['Author Keywords'] != '':
        for kw in row['Author Keywords'].split(';'):
            existing.add(kw.strip().lower())

    # Extrai todas as keywords que aparecem no texto de uma só vez
    found   = set(pattern.findall(text))
    new_kws = found - existing

    if not new_kws:
        return row['Author Keywords']

    added_total += len(new_kws)

    base = row['Author Keywords'] if pd.notna(row['Author Keywords']) else ''
    new_kws_str = '; '.join(new_kws)
    return f"{base}; {new_kws_str}" if base else new_kws_str

all_refs['Author Keywords'] = all_refs.apply(expand_keywords, axis=1)

media_depois = all_refs['Author Keywords'].apply(count_keywords).mean()

print(f"\nKeywords adicionadas ao dataset:          {added_total:,}")
print(f"Média de keywords por artigo ANTES:       {media_antes:.2f}")
print(f"Média de keywords por artigo DEPOIS:      {media_depois:.2f}")
print(f"Ganho médio por artigo:                   {media_depois - media_antes:.2f}")

Média de keywords por artigo ANTES: 0.83
Total de keywords únicas no dataset (>3 chars): 10,814

Keywords adicionadas ao dataset:          181,570
Média de keywords por artigo ANTES:       0.83
Média de keywords por artigo DEPOIS:      8.16
Ganho médio por artigo:                   7.33


In [98]:
all_refs.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24779 entries, 0 to 24778
Data columns (total 25 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Authors            7488 non-null   object 
 1   Author full names  7488 non-null   object 
 2   Author(s) ID       7488 non-null   object 
 3   Title              24779 non-null  object 
 4   Year               7507 non-null   float64
 5   Source title       24779 non-null  object 
 6   Volume             4839 non-null   object 
 7   Issue              4164 non-null   object 
 8   Art. No.           404 non-null    object 
 9   Page start         6952 non-null   object 
 10  Page end           6947 non-null   object 
 11  Cited by           7507 non-null   float64
 12  DOI                6293 non-null   object 
 13  Link               7507 non-null   object 
 14  Abstract           7507 non-null   object 
 15  Author Keywords    23958 non-null  object 
 16  Index Keywords     136

In [99]:
import pandas as pd

# ── Funções de detecção ────────────────────────────────────────────────────────

def has_human_action(references):
    if pd.isna(references) or references == '': return False
    for ref in references.split(';'):
        r = ref.lower()
        if 'mises' in r and 'human action' in r: return True
    return False

def has_theory_of_money(references):
    if pd.isna(references) or references == '': return False
    for ref in references.split(';'):
        r = ref.lower()
        if 'mises' in r and 'theory of money' in r: return True
    return False

def has_economic_calculation(references):
    if pd.isna(references) or references == '': return False
    for ref in references.split(';'):
        r = ref.lower()
        if 'mises' in r and 'economic calculation' in r: return True
    return False

def has_omnipotent_government(references):
    if pd.isna(references) or references == '': return False
    for ref in references.split(';'):
        r = ref.lower()
        if 'mises' in r and 'omnipotent government' in r: return True
    return False

def has_planning_for_freedom(references):
    if pd.isna(references) or references == '': return False
    for ref in references.split(';'):
        r = ref.lower()
        if 'mises' in r and 'planning for freedom' in r: return True
    return False

def has_theory_and_history(references):
    if pd.isna(references) or references == '': return False
    for ref in references.split(';'):
        r = ref.lower()
        if 'mises' in r and 'theory and history' in r: return True
    return False

def has_socialism(references):
    if pd.isna(references) or references == '': return False
    for ref in references.split(';'):
        r = ref.lower()
        if 'mises' in r and 'socialism' in r: return True
    return False

def has_road_to_serfdom(references):
    if pd.isna(references) or references == '': return False
    for ref in references.split(';'):
        r = ref.lower()
        if 'hayek' in r and 'road to serfdom' in r: return True
    return False

def has_use_of_knowledge(references):
    if pd.isna(references) or references == '': return False
    for ref in references.split(';'):
        r = ref.lower()
        if 'hayek' in r and 'use of knowledge' in r: return True
    return False

def has_constitution_of_liberty(references):
    if pd.isna(references) or references == '': return False
    for ref in references.split(';'):
        r = ref.lower()
        if 'hayek' in r and 'constitution of liberty' in r: return True
    return False

def has_interventionism(references):
    if pd.isna(references) or references == '': return False
    for ref in references.split(';'):
        r = ref.lower()
        if 'mises' in r and 'interventionism' in r: return True
    return False

def has_prices_and_production(references):
    if pd.isna(references) or references == '': return False
    for ref in references.split(';'):
        r = ref.lower()
        if 'hayek' in r and 'prices and production' in r: return True
    return False

def has_law_legislation_liberty(references):
    if pd.isna(references) or references == '': return False
    for ref in references.split(';'):
        r = ref.lower()
        if 'hayek' in r and 'law, legislation' in r: return True
    return False

def has_counter_revolution(references):
    if pd.isna(references) or references == '': return False
    for ref in references.split(';'):
        r = ref.lower()
        if 'hayek' in r and 'counter-revolution of science' in r: return True
    return False

def has_monetary_theory_trade_cycle(references):
    if pd.isna(references) or references == '': return False
    for ref in references.split(';'):
        r = ref.lower()
        if 'hayek' in r and 'monetary theory' in r: return True
    return False

def has_individualism_economic_order(references):
    if pd.isna(references) or references == '': return False
    for ref in references.split(';'):
        r = ref.lower()
        if 'hayek' in r and 'individualism' in r: return True
    return False

def has_epistemological_problems(references):
    if pd.isna(references) or references == '': return False
    for ref in references.split(';'):
        r = ref.lower()
        if 'mises' in r and 'epistemological' in r: return True
    return False

# ── Aplicação ──────────────────────────────────────────────────────────────────

all_refs['cite_mises_human_action']            = all_refs['References'].apply(has_human_action)
all_refs['cite_mises_theory_of_money']         = all_refs['References'].apply(has_theory_of_money)
all_refs['cite_mises_economic_calculation']    = all_refs['References'].apply(has_economic_calculation)
all_refs['cite_mises_omnipotent_government']   = all_refs['References'].apply(has_omnipotent_government)
all_refs['cite_mises_planning_for_freedom']    = all_refs['References'].apply(has_planning_for_freedom)
all_refs['cite_mises_theory_and_history']      = all_refs['References'].apply(has_theory_and_history)
all_refs['cite_mises_socialism']               = all_refs['References'].apply(has_socialism)
all_refs['cite_hayek_road_to_serfdom']         = all_refs['References'].apply(has_road_to_serfdom)
all_refs['cite_hayek_use_of_knowledge']        = all_refs['References'].apply(has_use_of_knowledge)
all_refs['cite_hayek_constitution_of_liberty'] = all_refs['References'].apply(has_constitution_of_liberty)
all_refs['cite_hayek_prices_and_production']   = all_refs['References'].apply(has_prices_and_production)
all_refs['cite_hayek_law_legislation_liberty'] = all_refs['References'].apply(has_law_legislation_liberty)
all_refs['cite_hayek_counter_revolution']      = all_refs['References'].apply(has_counter_revolution)
all_refs['cite_hayek_monetary_theory_trade_cycle'] = all_refs['References'].apply(has_monetary_theory_trade_cycle)
all_refs['cite_hayek_individualism_economic_order'] = all_refs['References'].apply(has_individualism_economic_order)
all_refs['cite_mises_epistemological_problems'] = all_refs['References'].apply(has_epistemological_problems)
all_refs['cite_mises_interventionism'] = all_refs['References'].apply(has_interventionism)


# ── Checagem ordenada por % ────────────────────────────────────────────────────

total = len(all_refs)
cite_cols = [col for col in all_refs.columns if col.startswith('cite_')]
results = sorted([(col, all_refs[col].sum()) for col in cite_cols], key=lambda x: x[1], reverse=True)

print(f"{'Obra':<50} {'N':>7} {'%':>7}")
print("-" * 66)
for col, n in results:
    print(f"{col:<50} {n:>7,} {n/total*100:>6.1f}%")

Obra                                                     N       %
------------------------------------------------------------------
cite_hayek                                          20,044   80.9%
cite_mises                                           7,507   30.3%
cite_hayek_use_of_knowledge                          7,293   29.4%
cite_hayek_road_to_serfdom                           6,313   25.5%
cite_hayek_law_legislation_liberty                   4,356   17.6%
cite_mises_human_action                              4,238   17.1%
cite_hayek_individualism_economic_order              3,308   13.4%
cite_hayek_constitution_of_liberty                   2,789   11.3%
cite_mises_socialism                                 1,136    4.6%
cite_hayek_prices_and_production                     1,002    4.0%
cite_mises_theory_of_money                             863    3.5%
cite_mises_economic_calculation                        863    3.5%
cite_hayek_counter_revolution                          690    

In [100]:
cite_cols = [col for col in all_refs.columns if col.startswith('cite_')]

results = []
for a in cite_cols:
    for b in cite_cols:
        if a == b:
            continue
        n_b       = all_refs[b].sum()
        n_a_and_b = (all_refs[a] & all_refs[b]).sum()
        if n_b == 0:
            continue
        p_a_given_b = n_a_and_b / n_b
        results.append((a, b, p_a_given_b, n_a_and_b, n_b))

results = sorted(results, key=lambda x: (x[1], -x[2]))

print(f"{'A (dado B)':<45} {'B':<45} {'P(A|B)':>8} {'n(A∩B)':>8} {'n(B)':>8}")
print("-" * 114)

prev_b = None
for a, b, p, n_ab, n_b in results:
    if b != prev_b:
        print()
        prev_b = b
    print(f"{a:<45} {b:<45} {p*100:>7.1f}% {n_ab:>8,} {n_b:>8,}")

A (dado B)                                    B                                               P(A|B)   n(A∩B)     n(B)
------------------------------------------------------------------------------------------------------------------

cite_hayek_use_of_knowledge                   cite_hayek                                       35.7%    7,156   20,044
cite_hayek_road_to_serfdom                    cite_hayek                                       31.0%    6,221   20,044
cite_hayek_law_legislation_liberty            cite_hayek                                       21.1%    4,228   20,044
cite_hayek_individualism_economic_order       cite_hayek                                       15.2%    3,055   20,044
cite_mises                                    cite_hayek                                       13.8%    2,772   20,044
cite_hayek_constitution_of_liberty            cite_hayek                                       13.0%    2,605   20,044
cite_mises_human_action                       cite_

In [101]:
from collections import Counter

cite_cols = [col for col in all_refs.columns if col.startswith('cite_')]

for col in cite_cols:
    subset = all_refs[all_refs[col] == True]['Author Keywords'].dropna()
    
    keyword_counts = Counter()
    for kw_string in subset:
        for kw in kw_string.split(';'):
            kw = kw.strip()
            if kw:
                keyword_counts[kw.lower()] += 1
    
    print(f"\n{'='*60}")
    print(f"{col} (n={all_refs[col].sum():,})")
    print(f"{'='*60}")
    print(f"{'Keyword':<45} {'N':>6} {'%':>7}")
    print("-" * 60)
    
    total_articles = all_refs[col].sum()
    for kw, count in keyword_counts.most_common(20):
        print(f"{kw:<45} {count:>6,} {count/total_articles*100:>6.1f}%")


cite_mises (n=7,507)
Keyword                                            N       %
------------------------------------------------------------
economic                                       2,135   28.4%
rights                                         1,846   24.6%
theory                                         1,584   21.1%
economics                                      1,526   20.3%
more                                           1,514   20.2%
social                                         1,442   19.2%
research                                       1,200   16.0%
analysis                                       1,184   15.8%
development                                      982   13.1%
science                                          956   12.7%
political                                        937   12.5%
work                                             915   12.2%
understanding                                    886   11.8%
business                                         823   11.0%
au